# Expectation value 

## Libraries 

In [1]:
# for creation of the matix 
import numpy as np  

# for the circuit itself
from qiskit.circuit.library import IQP
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp, random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator

# for api token 
import keyring as k3

print("Llibraries Loaded")


Llibraries Loaded


## Backend Section 

In [3]:
# set number of qubits for the IQP circuit, adjust as needed
n_qubits = 120  

In [5]:
# Get the API token from keyring
api_token = k3.get_password('ibm_quantum', 'api_token')


# set up your backend 
service = QiskitRuntimeService(channel="ibm_quantum", token=api_token)
backend = service.least_busy(
    operational=True, 
    simulator=False, 
    min_num_qubits=n_qubits
)

print(f"Using backend: {backend}")
print("Backend Loaded")


Using backend: <IBMBackend('ibm_kyiv')>
Backend Loaded


## Circuit Section 

In [6]:


# Create your IQP circuit and observable
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = IQP(mat)
observable = SparsePauliOp("Z" * n_qubits)  # Define the observable as Z^n_qubits for the expectation value
 
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

print("Circuit and Observable Created")


Circuit and Observable Created


## Run Section 

In [7]:
# create the job and run it

estimator = Estimator(mode=backend)
job = estimator.run([(isa_circuit, isa_observable)])
job_id = job.job_id()
print(f" > Job ID: {job_id}")
print("Job Created")


 > Job ID: czzt878d8drg008j4sg0
Job Created


## Post-Run Section 

In [8]:
# Wait for the job to complete and get the result
result = job.result()
 
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: 0.00390625
 > Metadata: {'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}
